# MarketPulse — Export for Power BI

Exports the key processed Delta tables as CSVs, to be loaded directly into Power BI Desktop for dashboard building. Power BI Desktop's free tier doesn't connect directly to Databricks without additional paid infrastructure, so CSV export is the simplest, most reliable path for this project.

In [0]:
tables_to_export = [
    "marketpulse_transactions",
    "marketpulse_customer_features",
    "marketpulse_experiments",
    "marketpulse_orders_clean"
]

for table in tables_to_export:
    df = spark.sql(f"SELECT * FROM {table}").toPandas()
    df.to_csv(f"/tmp/{table}.csv", index=False)
    print(f"{table}: {df.shape[0]} rows exported")

marketpulse_transactions: 110179 rows exported
marketpulse_customer_features: 93345 rows exported
marketpulse_experiments: 20000 rows exported
marketpulse_orders_clean: 99441 rows exported


## Download exported CSVs

Databricks' `/tmp/` directory is on cloud compute, not the local machine — files must be explicitly downloaded through Databricks' file browser or a direct download link.

In [0]:
%pip install boto3

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import boto3

# Upload processed CSVs to S3, then download from the AWS Console
# (avoids /dbfs FUSE mount, which isn't available on this serverless environment)

s3 = boto3.client(
    "s3",
    aws_access_key_id="ACCESS_KEY",
    aws_secret_access_key="SECRET_ACCESS_KEY",
    region_name="ap-southeast-2"
)

BUCKET = "marketpulse-narjeena-2026"

for table in tables_to_export:
    s3.upload_file(f"/tmp/{table}.csv", BUCKET, f"processed/{table}.csv")
    print(f"Uploaded: processed/{table}.csv")

Uploaded: processed/marketpulse_transactions.csv
Uploaded: processed/marketpulse_customer_features.csv
Uploaded: processed/marketpulse_experiments.csv
Uploaded: processed/marketpulse_orders_clean.csv
